# Bootstrapping the generation of JSON Schema

This actually reads the data from different datasets to generate the JSON Schema, so it should (and has been) run only once to build first version of the schemas, then manual or ad-hoc programmatic changes can be performed on the schemas. As the schemas will be used to validate data production and ingestion, generating it them from the data each time makes it a circular logical fallacy that can produce "wrong" data that auto-validate themselves. 

In [ ]:
import json
import glob
import os

def find_and_open_most_recent_json_file(file_prefix:str):
    files = glob.glob(f'{file_prefix}*json')
    files.sort(key=os.path.getmtime)
    found_file = files[-1]
    with open(found_file, 'r') as f:
        all_itms = json.load(f)
        return all_itms

# getting all the data needed for the bootstrapping from the Sommarioni, as it is the most complete dataset
hrs = find_and_open_most_recent_json_file('../1808_Sommarioni/data/sommarioni_hr_rde')
datasets = find_and_open_most_recent_json_file('../1808_Sommarioni/data/sommarioni_dataset_rde')
obs = find_and_open_most_recent_json_file('../1808_Sommarioni/data/sommarioni_obs_rde')
pois = find_and_open_most_recent_json_file('../1808_Sommarioni/data/sommarioni_poi_rde')
geos = find_and_open_most_recent_json_file('../1808_Sommarioni/data/sommarioni_geometries_rde')
map_and_layers = find_and_open_most_recent_json_file('../1808_Sommarioni/data/sommarioni_map_and_layers_')
map = [v for v in map_and_layers if v['rde_type'] == 'map'][0]
layers = [v for v in map_and_layers if v['rde_type'] == 'layer']
area = json.load(open('../areas/venice-area.geojson'))
config = json.load(open('../1808_Sommarioni/sommarioni_ds_configuration.json'))

In [ ]:
from genson import SchemaBuilder
SCHEMA_BASE_URL = "https://timemachine.epfl.ch"

def save_schema(build: SchemaBuilder):
    schema = build.to_schema()
    save_file = schema['$schema'].replace(SCHEMA_BASE_URL+'/', '')
    with open(save_file, 'w+') as f:
        json.dump(schema, f, indent=2)

In [ ]:
# bootstrapping the HR schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/hr.schema.json')
builder.add_schema({"type": "object", "properties": {}})
# TODO: customise the annotated_content field. 
hr_copy = {"rde_objects": []}
for hr in hrs['rde_objects']:
    t_hr = hr.copy()
    # metadata object is datset specific, we want to generate a generic schema, making it empty will avoid any validation issue. 
    t_hr['annotated_content'] = {}
    hr_copy['rde_objects'].append(t_hr)
builder.add_object(hr_copy)
save_schema(builder)

In [ ]:
# bootstrapping the obs schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/obs.schema.json')
builder.add_schema({"type": "object", "properties": {}})
builder.add_object(obs)

save_schema(builder)

In [ ]:
# bootstrapping the poi schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/poi.schema.json')
builder.add_schema({"type": "object", "properties": {}})
builder.add_object(pois)

save_schema(builder)

In [ ]:
# bootstrapping the geometries schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/geometries.schema.json')
builder.add_schema({"type": "object", "properties": {}})
builder.add_object(geos)

save_schema(builder)

In [ ]:
# bootstrapping the dataset schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/dataset.schema.json')
builder.add_schema({"type": "object", "properties": {}})
builder.add_object(datasets)

save_schema(builder)

In [ ]:
# bootstrapping the configuration schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/dataset_configuration.schema.json')
builder.add_schema({"type": "object", "properties": {}})
builder.add_object(config)

save_schema(builder)

In [ ]:
# bootstraping the map schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/map.schema.json')
builder.add_schema({"type": "object", "properties": {}})
builder.add_object(map)

save_schema(builder)

In [ ]:
# bootstrapping the layer schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/layer.schema.json')
builder.add_schema({"type": "object", "properties": {}})
for layer in layers:
    builder.add_object(layer)

save_schema(builder)

In [ ]:
# bootstrapping the area schema
builder = SchemaBuilder(f'{SCHEMA_BASE_URL}/area.schema.json')
builder.add_schema({"type": "object", "properties": {}})
builder.add_object(area)

save_schema(builder)